## 1. Imports

In [1]:
from pynq import Overlay, allocate, PL
import numpy as np

print("Imports OK")

Imports OK


## 2. Chargement de l'overlay

Place `multiplier_dma.bit` et `multiplier_dma.hwh` dans le même dossier que ce notebook.

In [2]:
PL.reset()
overlay = Overlay("design_1.bit")

print("Overlay chargé")
print("IP disponibles :")
for name in overlay.ip_dict:
    print(" -", name)

Overlay chargé
IP disponibles :
 - multiplier_dma_0
 - axi_dma_0
 - processing_system7_0


## 3. Détection du DMA et de l'IP

Cette cellule vérifie aussi que le DMA possède bien les deux canaux `sendchannel` et `recvchannel`.

In [3]:
dma = None
dma_name = None

for name in overlay.ip_dict:
    obj = getattr(overlay, name)

    if hasattr(obj, "sendchannel") and hasattr(obj, "recvchannel"):
        dma = obj
        dma_name = name
        break

if dma is None:
    raise RuntimeError(
        "Aucun AXI DMA avec MM2S + S2MM n'a été trouvé. "
        "Vérifie dans Vivado que Read Channel et Write Channel sont activés."
    )

mult = None
mult_name = None

for name in overlay.ip_dict:
    if "multiplier" in name.lower():
        mult = getattr(overlay, name)
        mult_name = name
        break

if mult is None:
    raise RuntimeError(
        "Aucune IP multiplier n'a été trouvée dans l'overlay."
    )

print("DMA détecté :", dma_name)
print("IP détectée  :", mult_name)
print("sendchannel  :", hasattr(dma, "sendchannel"))
print("recvchannel  :", hasattr(dma, "recvchannel"))

DMA détecté : axi_dma_0
IP détectée  : multiplier_dma_0
sendchannel  : True
recvchannel  : True


## 4. Registre `size`

In [4]:
registers = overlay.ip_dict[mult_name].get("registers", {})

print("Registres détectés :")
print(registers)

if "size" not in registers:
    raise KeyError(
        f"Registre 'size' introuvable. Registres disponibles : {list(registers.keys())}"
    )

SIZE_OFFSET = registers["size"]["address_offset"]
CTRL_OFFSET = 0x00

print("SIZE_OFFSET =", hex(SIZE_OFFSET))

Registres détectés :
{'CTRL': {'address_offset': 0, 'size': 32, 'access': 'read-write', 'description': 'Control signals', 'fields': {'AP_START': {'bit_offset': 0, 'bit_width': 1, 'access': 'read-write', 'description': "Control signal Register for 'ap_start'."}, 'AP_DONE': {'bit_offset': 1, 'bit_width': 1, 'access': 'read-only', 'description': "Control signal Register for 'ap_done'."}, 'AP_IDLE': {'bit_offset': 2, 'bit_width': 1, 'access': 'read-only', 'description': "Control signal Register for 'ap_idle'."}, 'AP_READY': {'bit_offset': 3, 'bit_width': 1, 'access': 'read-only', 'description': "Control signal Register for 'ap_ready'."}, 'RESERVED_1': {'bit_offset': 4, 'bit_width': 3, 'access': 'read-only', 'description': 'Reserved.  0s on read.'}, 'AUTO_RESTART': {'bit_offset': 7, 'bit_width': 1, 'access': 'read-write', 'description': "Control signal Register for 'auto_restart'."}, 'RESERVED_2': {'bit_offset': 8, 'bit_width': 1, 'access': 'read-only', 'description': 'Reserved.  0s on read

## 5. Fonction de traitement FPGA

Le buffer d'entrée contient `2×N` entiers :

`A0, B0, A1, B1, ...`

Le buffer de sortie contient `N` résultats.

In [5]:
def multiply_vectors_fpga(A, B):
    A = np.asarray(A, dtype=np.int32)
    B = np.asarray(B, dtype=np.int32)

    if A.shape != B.shape:
        raise ValueError("A et B doivent avoir la même taille.")

    N = len(A)

    in_buffer = allocate(shape=(2 * N,), dtype=np.int32)
    out_buffer = allocate(shape=(N,), dtype=np.int32)

    try:
        in_buffer[0::2] = A
        in_buffer[1::2] = B

        in_buffer.flush()
        out_buffer.flush()

        mult.write(SIZE_OFFSET, N)

        # Préparer la réception avant l'envoi
        dma.recvchannel.transfer(out_buffer)

        # Démarrer l'IP HLS
        mult.write(CTRL_OFFSET, 0x01)

        # Envoyer A0, B0, A1, B1, ...
        dma.sendchannel.transfer(in_buffer)

        dma.sendchannel.wait()
        dma.recvchannel.wait()

        out_buffer.invalidate()

        return np.array(out_buffer, dtype=np.int32)

    finally:
        in_buffer.freebuffer()
        out_buffer.freebuffer()

## 6. Premier test

In [6]:
A = np.array([1, 2, 3, 4, 5], dtype=np.int32)
B = np.array([10, 20, 30, 40, 50], dtype=np.int32)

result_hw = multiply_vectors_fpga(A, B)
result_sw = A * B

print("A        =", A)
print("B        =", B)
print("FPGA     =", result_hw)
print("Software =", result_sw)
print("Correct  =", np.array_equal(result_hw, result_sw))

A        = [1 2 3 4 5]
B        = [10 20 30 40 50]
FPGA     = [ 10  40  90 160 250]
Software = [ 10  40  90 160 250]
Correct  = True


## 7. Test aléatoire

In [7]:
rng = np.random.default_rng(seed=42)

N = 1024

A = rng.integers(-100, 101, size=N, dtype=np.int32)
B = rng.integers(-100, 101, size=N, dtype=np.int32)

result_hw = multiply_vectors_fpga(A, B)
result_sw = A * B

correct = np.array_equal(result_hw, result_sw)

print("Nombre de multiplications :", N)
print("Résultats corrects        :", correct)

print("\nPremières valeurs :")
print("A        :", A[:10])
print("B        :", B[:10])
print("FPGA     :", result_hw[:10])
print("Software :", result_sw[:10])

if not correct:
    errors = np.where(result_hw != result_sw)[0]
    print("\nNombre d'erreurs :", len(errors))
    print("Premiers indices incorrects :", errors[:10])

Nombre de multiplications : 1024
Résultats corrects        : True

Premières valeurs :
A        : [-83  55  31 -12 -13  72 -83  40 -60 -82]
B        : [ 65 -89 -85 -44  13 -33  -3 -66  46 -37]
FPGA     : [-5395 -4895 -2635   528  -169 -2376   249 -2640 -2760  3034]
Software : [-5395 -4895 -2635   528  -169 -2376   249 -2640 -2760  3034]
